# Les Emeutes

## Descrition
On analyse souvent les émeutes comme une explosion soudaine et imprévisible. Est ce vraiment le cas ? Il est possible que les territoires avec un taux d'inactivité plus élevé subissent plus frequemment ces désagréments. L'objectif de ce projet Python est ainsi d'explorer les dynamiques des émeutes et d'identifier les principaux drivers qui accompagnent ces évenements.

À travers un schema de collecte, de nettoyage et d'analyse de données, nous étudirons les corrélations entre présence de quartiers prioritaires (QPV), le taux d'inactivité et les émeutes. 

La difficulté première du projet étant d'obtenir des renseignements fiables sur la présence d'emeutes en France, nous avons choisi de croiser deux sources de données complémentaires: GDELT, qui capture l'activité médiatique et événementielle mondiale en temps réel, et Google Trends, qui reflète l'attention directe des citoyens à travers leurs requêtes de recherche.

## Problématique

## Plan:

- Récupération des données:
    - GDELT
    - Google Trends
- Nettoyage
- Comparaison Gdelt, Google Trends
- Les données sociales:
    - Taux d'Inactivité
    - Quartier prioritaires
- Peut-on trouver une relation ?
- Limites et Conclusion


## Récupération des données

Deux bases de données sont utilisées pour caractériser au mieux ce phénomène: GDELT et Google Trends. Avant de les récupérer, importons les librairies



In [ ]:
#import google-cloud-bigquery db-dtypes ## Non natif, pour ...
import json
import s3fs

from GT_data.Google_trends import Google_Trends # Créée par nous même
from trendspy import Trends ## Non natif, pour scraper les trends

import pandas as pd
import matplotlib.pyplot as plt

-> GDELT:

Descritption: GDELT est une base de données ouverte qui scanne en continu les médias du monde entier (presse en ligne, agences, web) et transforme chaque article en événement structuré. 

Objectif: La parution d'articles sur le sujet des émeutes est le média le plus précis afin de déterminer où et quand se sont déroulés ces évenements.

Nous prendrons deux sous-bases de GDELT: une petite, précise mais avec peu de lignes; et une grosse necessitant un nettoyage

In [ ]:
# --------- GDELT ---------------------

# -- Settings --
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})
MY_BUCKET = "vgounot"

# -- Import --
FILE_PATH_S3 = f"{MY_BUCKET}/diffusion/gcloud_emeutes_all.json"
with fs.open(FILE_PATH_S3, 'r', encoding="utf-8") as file_in:
    df_GDELT_small = pd.read_json(file_in)
    print("GDELT bigQuery small récupéré")
    print(df_GDELT_small.size)


FILE_PATH_S3 = f"{MY_BUCKET}/diffusion/2023_gcloud_emeutes.parquet"

with fs.open(FILE_PATH_S3, 'rb', encoding="utf-8") as file_in:
    df_GDELT_large = pd.read_parquet(file_in)
    print("GDELT bigQuery gros récupéré")
    print(df_GDELT_large.size)



-> Google Trends

Description: Google Trends permet de connaitre le nombre de requêtes pour un mot clef.
Objectif: avoir un aperçu temporel du ressenti d'émeutes en France. 
Dans un but de vérifier les informations récupérées avec GDELT, Google Trends sert à caractériser l'intérêt des francais pour les emeutes à travers des recherches comme "Mortier", "LBD", "casseurs", ... 



In [ ]:
G_trend = Google_Trends()

# Mots de Recherche
mots_contexte = ["Emeute", "violences urbaines"]
mots_acteurs = ["casseurs"]
mots_actions = ["saccage", 'pillage']
mots_matériel = ["mortier", "LBD"]
mots_de_recherche = mots_contexte+mots_acteurs + mots_actions + mots_matériel

# Récupération des données, normalisation et création d'une moyenne de recherche
df = G_trend.get_trends(mots_de_recherche, geo="FR", timeframe="today 3-m")
df = G_trend.normalisation(df)
df = G_trend.add_average(df)

df_google_trends = df[["mean", "Time"]]

G_trend.plot(df_google_trends)

## Comparer GDELT et Google Trends

Maintenant que nous avons les deux bases d'émeutes, nous pouvons mettre en perspective les résultats trouvés. Les évenements trouvés sur GDELT étant discrets, pour avoir une meilleure représentation des périodes d'émeutes, nous allons aggréger le nombre d'articles parus sur des périodes de 3 jours.

In [ ]:
def comparer_GDELT_Gtrend(G_trend_class, df_gdelt, aggregation_periode = "3D", afficher=True):
    # Pour aggréger par x jours ( mettre "xD" en periode ou 'W-SUN' pour une semaine)
    df_gdelt_weekly = (
        df_gdelt
        .resample(aggregation_periode, on='Time')
        .size()
        .reset_index(name='nb_emeutes')
    )

    if afficher: G_trend_class.comparer_GDELT_plot(df_gdelt_weekly)

    return df_gdelt_weekly

# Bien définir la date
if not "Time"  in df_GDELT_small.columns:
    df_GDELT_small = df_GDELT_small.rename(columns={'observed_at': 'Time'})
    df_GDELT_small['Time'] = pd.to_datetime(df_GDELT_small['Time']).dt.tz_localize(None)


df_gdelt_weekly_small = comparer_GDELT_Gtrend(G_trend, df_GDELT_small, "3D")

Pour comparer à la table entière, sur un intervalle de 4 ans, on utilise la même fonction:

In [ ]:
# On doit récupérer à nouveau les données sur plus d'années
G_trend_large = Google_Trends()


# Récupération des données, normalisation et création d'une moyenne de recherche
df_large = G_trend_large.get_trends(mots_de_recherche, geo="FR", timeframe="today 4-y")
df_large = G_trend_large.normalisation(df_large)
df_large = G_trend_large.add_average(df_large)

df_google_trends_large = df_large[["mean", "Time"]]

G_trend_large.plot(df_google_trends_large) # on affiche ce résultat

In [ ]:
# Comparer
df_GDELT_large['Time'] = pd.to_datetime(df_GDELT_large['SQLDATE'].astype(str), format='%Y%m%d')
df_gdelt_weekly_large = comparer_GDELT_Gtrend(G_trend_large, df_GDELT_large, "3D")

### Remarque
On aperçoit bien une corrélation entre les résultats des données de recherche des utilisateurs (Google Trends) et le nombre de parutions dans les journeaux d'articles mentionnant des émeutes (GDELT). Trois périodes se distinguent: entre mars et août 2023 et fin 2026. Le premier correspond aux mouvements contre la réforme et surtout aux durcissement des mobilisations pendant la periode de mars 2023. Le second correspond aux évenements survenus après la mort de Nahel Merzouk en juin 2023. Le dernier est lié aux mouvements étudiants.

Si on met maintenant les graphiques de Google Trends et les deux graphiques de GDELT à côté, on obtient:

In [ ]:
df_GDELT_large_filtre = df_gdelt_weekly_large[df_gdelt_weekly_large["Time"] > "2026-08-15"]

# Axe de gauche : les deux bases GDELT
plt.plot(df_gdelt_weekly_small["Time"], df_gdelt_weekly_small["nb_emeutes"], label="GDELT Précis")
plt.plot(df_GDELT_large_filtre["Time"], df_GDELT_large_filtre["nb_emeutes"], label="GDELT ALL")
plt.ylabel("Nb émeutes (GDELT)")
plt.legend(loc="upper left")

# Axe de droite : Google Trends
plt.twinx()
plt.plot(df_google_trends["Time"], df_google_trends["mean"], color="red", linestyle="--", label="Google_trends")
plt.ylabel("Google Trends")
plt.legend(loc="upper right")

plt.title("Comparaison Google_trends et les deux bases GDELT")
plt.show()

### Remarques

La base plus précise s'accorde mieux avec les résultats google Trends, notamment au niveau du premier pic; mais ignore le second et comprend bien moins de données. 

Comparons, sur la même période le nombre de points :

In [ ]:
nSmall = df_GDELT_small[df_GDELT_small["Time"]> "2026-08-15"].size
nLarge = df_GDELT_large[df_GDELT_large["Time"]> "2026-08-15"].size
print("nb données pour GDELT précis:", nSmall)
print("nb données pour GDELT all:", nLarge)
print("C'est", round(nLarge/nSmall * 100)/100, "fois plus")

## Données sociales

## Peut on trouver une relation ?

## Limites et Conclusion 

Avant de conclure cette étude, nuançons les résultats trouvés et détaillons les limites de ce travail.

1. Des données implicites: \
La difficulté de ce travail a été de modéliser les émeutes en France. Aucune base de donnée libre d'accès recense fidèlement ces évènements. Nous avons donc reconstruits des estimateurs à partir des publications dans les médias et des recherches des utilisateurs. Néanmoins, ces mesures restent imparfaites.

2. Des analyses simplifiées: \
Nous savons que d'autres variables géographiques et sociales peuvent expliquer les émeutes comme la perception d'inégalités, le pouvoir d'achat moyen dans un territoire... Les résulats trouvés pourraient se révéler provenant d'une corrélation plus que d'une causalité. 

